# Level 3 Second Term Project

## Machine Learning GitHub Repository Analysis

**Student Name:** Mazen  
**Student ID:** STUDENT_ID  
**Date:** 22 August 2026

### Project Overview
This project collects public GitHub repository data about machine learning, cleans the data with Pandas, stores it in SQLite, analyzes it with SQL, and creates simple charts with Matplotlib.

## Task 1 - Data Collection and Preparation

### 1. Import Libraries

In [ ]:
import requests
import pandas as pd

### 2. Collect Repository Data from the GitHub API

In [ ]:
url = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc&per_page=100"

response = requests.get(url)
print("Status code:", response.status_code)

data = response.json()
print("Repositories received:", len(data["items"]))

The API request returns the repository data in JSON format. The next step is to place the repository records into a Pandas DataFrame.

### 3. Create and Explore the DataFrame

In [ ]:
df = pd.DataFrame(data["items"])
df.head()

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
df.info()

### 4. Select the Required Columns

In [ ]:
required_columns = [
    "name",
    "owner",
    "language",
    "stargazers_count",
    "forks_count",
    "watchers_count",
    "open_issues_count",
    "created_at",
    "updated_at",
    "license"
]

df = df[required_columns]
df.head()

### 5. Extract Values from Nested Columns

In [ ]:
df["owner"] = df["owner"].apply(lambda x: x.get("login") if isinstance(x, dict) else None)
df["license"] = df["license"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)

df.head()

### 6. Check and Handle Missing Values

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

In [ ]:
df["language"] = df["language"].fillna("Unknown")
df["license"] = df["license"].fillna("Unknown")

print("Missing values after cleaning:")
print(df.isnull().sum())

### 7. Check and Remove Duplicate Records

In [ ]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate rows after cleaning:", df.duplicated().sum())

### 8. Prepare the Date Columns and Rename Columns

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"])
df["updated_at"] = pd.to_datetime(df["updated_at"])

df = df.rename(columns={
    "stargazers_count": "stars",
    "forks_count": "forks",
    "watchers_count": "watchers",
    "open_issues_count": "open_issues",
    "created_at": "created_date",
    "updated_at": "updated_date"
})

df.head()

In [ ]:
df.info()

### 9. Save and Verify the Cleaned Dataset

In [ ]:
df.to_csv("github_projects.csv", index=False)
print("Saved github_projects.csv")

In [ ]:
check_df = pd.read_csv("github_projects.csv")
print("Rows in saved file:", len(check_df))
print("Columns in saved file:", list(check_df.columns))
check_df.head()

## Task 2 - Store and Analyse the Data

### 1. Create the SQLite Database

In [ ]:
import sqlite3

conn = sqlite3.connect("github_projects.db")
df = pd.read_csv("github_projects.csv")
df.to_sql("Repositories", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM Repositories LIMIT 5", conn)

### 2. Filtering and Searching

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE stars > 10000;
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE name LIKE '%Machine%';
"""
pd.read_sql(query, conn)

### 3. Logical Operators: AND, OR, NOT

In [ ]:
query = """
SELECT name, stars, forks
FROM Repositories
WHERE stars > 1000
AND forks > 100;
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT name, language, stars
FROM Repositories
WHERE language = 'Python'
OR language = 'C++';
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT name, language, stars
FROM Repositories
WHERE NOT language = 'Python';
"""
pd.read_sql(query, conn)

### 4. Sorting and Top 10 Repositories

In [ ]:
query = """
SELECT name, stars
FROM Repositories
ORDER BY stars DESC;
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT name, stars
FROM Repositories
ORDER BY stars DESC
LIMIT 10;
"""
top10 = pd.read_sql(query, conn)
top10

### 5. Aggregate Functions

In [ ]:
query = """
SELECT COUNT(*) AS total_repositories
FROM Repositories;
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT AVG(stars) AS average_stars
FROM Repositories;
"""
pd.read_sql(query, conn)

### 6. Grouping Analysis

In [ ]:
query = """
SELECT language, COUNT(*) AS repository_count
FROM Repositories
GROUP BY language
HAVING COUNT(*) > 5
ORDER BY repository_count DESC;
"""
language_groups = pd.read_sql(query, conn)
language_groups

### 7. Create Visualizations

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top10["name"], top10["stars"])
plt.xlabel("Stars")
plt.ylabel("Repository")
plt.title("Top 10 Machine Learning Repositories by Stars")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
df["created_date"] = pd.to_datetime(df["created_date"])
yearly = df.groupby(df["created_date"].dt.year).size()

plt.figure(figsize=(10, 6))
plt.plot(yearly.index, yearly.values, marker="o")
plt.xlabel("Year")
plt.ylabel("Number of Repositories")
plt.title("Repository Creation Trend")
plt.show()

### 8. Results and Findings

After running the analysis, review the tables and charts above. The most popular repositories are the ones with the highest star counts. The programming-language grouping shows which languages appear most often in the dataset. The creation trend shows how the number of repositories in the search results changes across years.

These findings should be read together with the actual SQL outputs and charts because the values can change as GitHub data changes.

## Task 3 - Git, GitHub, and Ethics

### GitHub Repository

**GitHub Repository:** PASTE_YOUR_GITHUB_REPOSITORY_LINK_HERE

## Ethics Reflection

### Question 1: Why is it important to verify data collected from public APIs?

It is important to verify API data because it can contain missing, incorrect, or unexpected values. Checking the data helps make sure that the analysis uses reliable information.

### Question 2: Why should data analysts document the source of their data?

Data analysts should document the source so other people can see where the data came from and check it when needed. It also makes the analysis easier to understand and repeat.

### Question 3: How can missing or inaccurate data affect data analysis and decision-making?

Missing or inaccurate data can change calculations and charts and may lead to wrong conclusions. Cleaning and checking the data before analysis helps reduce this problem.